In [18]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

california = fetch_california_housing()

df = pd.DataFrame(california.data, columns=california.feature_names)

df['Target_Price'] = california.target

# Elimino i campioni facili da predire
#df = df[df['Target_Price']<5.0]

print(df.info())

print("--- PRIME 5 RIGHE ---")
print(df.head())

print("\n--- CORRELAZIONE CON IL PREZZO ---")
print(df.corr()['Target_Price'].sort_values(ascending=False))

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   MedInc        20640 non-null  float64
 1   HouseAge      20640 non-null  float64
 2   AveRooms      20640 non-null  float64
 3   AveBedrms     20640 non-null  float64
 4   Population    20640 non-null  float64
 5   AveOccup      20640 non-null  float64
 6   Latitude      20640 non-null  float64
 7   Longitude     20640 non-null  float64
 8   Target_Price  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB
None
--- PRIME 5 RIGHE ---
   MedInc  HouseAge  AveRooms  ...  Latitude  Longitude  Target_Price
0  8.3252      41.0  6.984127  ...     37.88    -122.23         4.526
1  8.3014      21.0  6.238137  ...     37.86    -122.22         3.585
2  7.2574      52.0  8.288136  ...     37.85    -122.24         3.521
3  5.6431      52.0  5.817352  ...     37.85    -122.25         3.413
4  3.8462    

In [19]:
from sklearn.model_selection import train_test_split

features = df.drop(columns=['Target_Price'])

x_train, x_test, y_train, y_test = train_test_split(features, df['Target_Price'], test_size=0.3, random_state=42)

In [20]:
import os
n_jobs = max(1, os.cpu_count() // 2)

from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score

# No tuning
ini_model = XGBRegressor(objective="reg:squarederror", n_estimators=300, random_sate=42, n_jobs=n_jobs)
ini_model.fit(x_train, y_train)

cv_train = cross_val_score(ini_model, x_train, y_train, cv=5, scoring='r2').mean()
score_test = ini_model.score(x_test, y_test)
print(f"CV train: {cv_train:.4f} | Test: {score_test:.4f}")

CV train: 0.8250 | Test: 0.8393


In [21]:
# With tuning
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True), # Weight del singolo albero durante la correzione sequenziale, log distribuisce i trial in modo che l'ordine di grandezza riceva la stessa attenzione (0.005-0.05, 0.05-0.15, 0.15-3)
        'max_depth': trial.suggest_int('max_depth', 3, 10), # Di meno rispetto al random, 4-6 top, deve andare a correggere gli errori del precedente, quindi meno profondità
        'n_estimators': trial.suggest_int('n_estimators', 100, 1500), # numero di alberi superiore del random, ha bisogno di più alberi in sequenza per essere ottimale
        'subsample': trial.suggest_float('subsample', 0.5, 1.0), # frazione del set casuale dato in pasto ai singoli alberi
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0), # frazione di feature casuale per albero
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 40), # uguale ai minimi sample per foglia nel random forest
        'gamma': trial.suggest_float('gamma', 0.0, 5.0), # guadagno minimo che uno split deve produrre per essere eseguito
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),   # L1: penalità sul valore assoluto dei pesi foglia → azzera i pesi irrilevanti (feature selection implicita)
        # la penalità è il valore assoluto invece del quadrato. Matematicamente questa differenza sembra piccola ma ha un effetto completamente diverso in pratica.
        # Con la penalità sul valore assoluto, per XGBoost conviene portare i pesi irrilevanti esattamente a zero piuttosto che tenerli piccoli. Questo perché azzerare completamente un peso elimina tutta la sua penalità, mentre tenerlo anche solo a 0.001 costa comunque qualcosa.
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True)  # L2: penalità sul quadrato dei pesi foglia → schiaccia tutti i pesi verso zero senza azzerarli (default=1)
        # Aggiunge alla funzione di loss un termine extra pari alla somma dei quadrati di tutti i pesi foglia. Questo penalizza i pesi grandi durante l'ottimizzazione, costringendo XGBoost a preferire pesi più piccoli e moderati. Il risultato è un modello più conservativo che non si fida troppo di nessuna singola foglia.
        # Il quadrato fa sì che pesi molto grandi vengano penalizzati sproporzionatamente — un peso di 10 costa 100 in penalità, un peso di 2 costa solo 4. Tutti i pesi sopravvivono ma vengono compressi.
    }   

    model = XGBRegressor(**params, objective="reg:squarederror", random_state=42, n_jobs=n_jobs)
    return cross_val_score(model, x_train, y_train, cv=5, scoring='r2').mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=150, show_progress_bar=True)

print(f'Best params: {study.best_params}')

print(f'Best score: {study.best_value}')

Best trial: 139. Best value: 0.848306: 100%|██████████| 150/150 [08:56<00:00,  3.58s/it]

Best params: {'learning_rate': 0.01831433662139489, 'max_depth': 10, 'n_estimators': 1346, 'subsample': 0.7255830825137229, 'colsample_bytree': 0.7132023445298896, 'min_child_weight': 26, 'gamma': 0.06962743328501877, 'reg_alpha': 0.0011479404863894988, 'reg_lambda': 3.410480336807639e-05}
Best score: 0.8483058017592711


In [22]:
best_model = XGBRegressor(**study.best_params, n_jobs=n_jobs, random_state=42)
best_model.fit(x_train, y_train)
best_model.score(x_test, y_test)

0.8576063770380004